# Chapter 6 — Multimodal Generation

This notebook performs autoregressive generation:

    image + prompt → generated text

We use:
- frozen LM
- trained projector
- greedy decoding

No training here.

## Generation Strategy

We follow standard autoregressive decoding:

1. Build multimodal prefix:
      [image tokens] + [prompt tokens]

2. Forward pass → logits

3. Take next token

4. Append token

5. Repeat

In [1]:
import torch
import torch.nn.functional as F

## Setup

Assume:
- vision_encoder
- projector
- lm
- tokenizer

In [2]:
BATCH = 1
NUM_PATCHES = 64
LM_DIM = 1024
VOCAB_SIZE = 10000
MAX_NEW_TOKENS = 10

In [3]:
class DummyLM(torch.nn.Module):
    def __init__(self, dim, vocab):
        super().__init__()
        self.embed = torch.nn.Embedding(vocab, dim)
        self.lm_head = torch.nn.Linear(dim, vocab)

    def forward(self, embeds):
        return self.lm_head(embeds)

lm = DummyLM(LM_DIM, VOCAB_SIZE)

In [4]:
vision_encoder = lambda x: torch.randn(BATCH, NUM_PATCHES, 512)
projector = torch.nn.Linear(512, LM_DIM)

## Build Image Prefix

We build once.
It remains fixed during generation.

In [5]:
images = torch.randn(BATCH, 3, 224, 224)

vision_feats = vision_encoder(images)
vision_embeds = projector(vision_feats)

im_start = torch.zeros(BATCH, 1, LM_DIM)
im_end   = torch.zeros(BATCH, 1, LM_DIM)

image_block = torch.cat([im_start, vision_embeds, im_end], dim=1)
image_len = image_block.shape[1]

## Prompt Tokens

We simulate prompt tokens.

In [6]:
prompt_ids = torch.randint(0, VOCAB_SIZE, (BATCH, 4))
generated_ids = prompt_ids.clone()

## Autoregressive Generation Loop

In [7]:
for step in range(MAX_NEW_TOKENS):
    # Embed current tokens
    text_embeds = lm.embed(generated_ids)

    # Build joint sequence
    joint_embeds = torch.cat([image_block, text_embeds], dim=1)

    # Forward
    logits = lm(joint_embeds)

    # Take last text token logits
    text_logits = logits[:, image_len:, :]
    next_token_logits = text_logits[:, -1, :]

    # Greedy selection
    next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

    # Append
    generated_ids = torch.cat([generated_ids, next_token], dim=1)

## Generated Token IDs

In [8]:
print("Generated IDs shape:", generated_ids.shape)
print(generated_ids)

Generated IDs shape: torch.Size([1, 14])
tensor([[3042, 2561, 6971,  229, 3032, 1783, 4373,  862, 8132, 2393, 1770, 7361,
         2347, 6957]])


## Sampling Variant

Replace argmax with temperature sampling.

In [9]:
temperature = 0.8
probs = F.softmax(next_token_logits / temperature, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)

## What This Confirms

✔ Image embeddings remain fixed prefix  
✔ Text grows autoregressively  
✔ LM produces logits conditioned on image  
✔ Multimodal inference works